# LLM Bot Arena — Gemini Interactive Notebook

This notebook replaces the previous Python scripts and gives you an end-to-end interactive workflow to:

1. create a Gemini bot profile,
2. validate it with a test call,
3. save/load profile JSON under `bots/`, and
4. run an interactive multi-turn chat loop.

Use this notebook as the primary tool for the workflow described in the README.


## Prerequisites

- A valid Gemini API key
- Internet access
- Python kernel with standard library support

> Your API key is never written to disk by this notebook unless you explicitly add code to do so.


In [ ]:
from __future__ import annotations

import getpass
import json
import os
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib import error, request

API_BASE = "https://generativelanguage.googleapis.com/v1beta"
DEFAULT_MODEL = "gemini-2.0-flash"
BOTS_DIR = Path("bots")
BOTS_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class BotProfile:
    name: str
    model: str
    system_prompt: str
    created_at_utc: str


## Gemini HTTP helper

This helper performs a `generateContent` call with a system instruction and either a single user message or full conversation history.


In [ ]:
def call_gemini(api_key: str, model: str, system_prompt: str, contents: list[dict[str, Any]]) -> str:
    url = f"{API_BASE}/models/{model}:generateContent?key={api_key}"
    payload = {
        "system_instruction": {"parts": [{"text": system_prompt}]},
        "contents": contents,
    }

    data = json.dumps(payload).encode("utf-8")
    req = request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with request.urlopen(req, timeout=90) as resp:
            body = resp.read().decode("utf-8")
    except error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Gemini API error ({exc.code}): {detail}") from exc
    except error.URLError as exc:
        raise RuntimeError(f"Network error calling Gemini API: {exc}") from exc

    parsed: dict[str, Any] = json.loads(body)
    candidates = parsed.get("candidates") or []
    if not candidates:
        raise RuntimeError(f"No candidates in Gemini response: {body}")

    parts = candidates[0].get("content", {}).get("parts", [])
    texts = [part.get("text", "") for part in parts if isinstance(part, dict)]
    response_text = "\n".join(t for t in texts if t).strip()
    if not response_text:
        raise RuntimeError(f"Empty text response from Gemini: {body}")
    return response_text


## Step 1 — Configure bot profile inputs


In [ ]:
BOT_NAME = "parable_guide"
MODEL = DEFAULT_MODEL
SYSTEM_PROMPT = """A reflective advisor who helps users think deeply.

Rules:
- Use concise responses.
- Ask at least one follow-up question.
- Avoid absolute statements.
"""
OUTPUT_PATH = BOTS_DIR / f"{BOT_NAME}.json"

print("BOT_NAME:", BOT_NAME)
print("MODEL:", MODEL)
print("OUTPUT_PATH:", OUTPUT_PATH)


## Step 2 — Provide API key for this session


In [ ]:
API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass.getpass("Gemini API key (input hidden): ").strip()
if not API_KEY:
    raise RuntimeError("No API key provided.")

print("API key loaded for this notebook session.")


## Step 3 — Validate bot behavior


In [ ]:
validation_prompt = "Briefly introduce yourself in one sentence."
validation_reply = call_gemini(
    api_key=API_KEY,
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    contents=[{"role": "user", "parts": [{"text": validation_prompt}]}],
)
print("Validation reply:\n")
print(validation_reply)


## Step 4 — Save bot profile JSON


In [ ]:
profile = BotProfile(
    name=BOT_NAME,
    model=MODEL,
    system_prompt=SYSTEM_PROMPT.strip(),
    created_at_utc=datetime.now(timezone.utc).isoformat(),
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(asdict(profile), f, indent=2)
    f.write("\n")

print(f"Saved profile -> {OUTPUT_PATH}")


## Step 5 — Load an existing bot profile


In [ ]:
profiles = sorted(BOTS_DIR.glob("*.json"))
if not profiles:
    raise RuntimeError("No profiles found in ./bots. Run the save step first.")

for i, p in enumerate(profiles, start=1):
    print(f"{i}. {p}")

selection = input("Choose profile number [1]: ").strip() or "1"
index = int(selection)
if index < 1 or index > len(profiles):
    raise RuntimeError("Selection out of range.")

selected_profile_path = profiles[index - 1]
selected_profile = json.loads(selected_profile_path.read_text(encoding="utf-8"))

for key in ("name", "model", "system_prompt"):
    if key not in selected_profile:
        raise RuntimeError(f"Profile missing required key: {key}")

print(f"Loaded profile: {selected_profile_path}")
print("Bot:", selected_profile["name"])
print("Model:", selected_profile["model"])


## Step 6 — Interactive chat loop (type `/exit` to quit)


In [ ]:
conversation: list[dict[str, Any]] = []
name = selected_profile["name"]
model = selected_profile["model"]
system_prompt = selected_profile["system_prompt"]

print(f"Starting chat with {name} using {model}. Type /exit to quit.")

while True:
    user_text = input("you> ").strip()
    if not user_text:
        continue
    if user_text.lower() in {"/exit", "exit", "quit"}:
        print("Goodbye.")
        break

    conversation.append({"role": "user", "parts": [{"text": user_text}]})
    try:
        reply = call_gemini(
            api_key=API_KEY,
            model=model,
            system_prompt=system_prompt,
            contents=conversation,
        )
    except RuntimeError as exc:
        print(f"error> {exc}")
        conversation.pop()
        continue

    print(f"{name}> {reply}\n")
    conversation.append({"role": "model", "parts": [{"text": reply}]})


## Optional: one-shot helper


In [ ]:
def ask_once(prompt: str) -> str:
    return call_gemini(
        api_key=API_KEY,
        model=selected_profile["model"],
        system_prompt=selected_profile["system_prompt"],
        contents=[{"role": "user", "parts": [{"text": prompt}]}],
    )

# Example:
# print(ask_once("Give me a short plan to improve my writing habits."))
